# Tables gen (.tex)

In [1]:
import pandas as pd
import torch
from torch.utils.data import TensorDataset, ConcatDataset
from data_utils import extract_TensorDataset, extract_boundary
from load_store_utils import resume_model
from typing import List
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

PDE = "AllenCahn"
ACTUAL_MODE = "PINN"
LR = "_CosLrAnnealing"
MEMORY = "LongMemory"
BD = "_LocalFullBd" # _LocalFullBd _LocalBd
LIMITED_MEMORY = True
N_EPOCHS = 200
DWA_MODE = "Norm1"
PARAM_INDEX = None
CLIP = "GradClip"

FONT = "Cambria !important"
FONT_SIZE = "12pt !important"

YELLOW = ["#FFF3D1", "#FFE7A5", "#FFD483", "#B36430"]
ORANGE = ["#FFE6B8", "#FFD8B8", "#FFBF8A", "#D15E00", "#A34900"]
PURPLE = ["#EDE6FF", "#C9BCEA", "#A48BDE", "#6500A3"]
BROWN = ["#E8CECE", "#D9AFAF", "#BC7171", "#8E4343"]
RED = ["#FFD0D0", "#D59999", "#D17878", "#8E4343"]
BLUE = ["#E6E9FF", "#B8C2FF", "#687FFF", "#4D5AD0"]#"#8799FF"
GRAY = ["#E5E5E5", "#968C8C", "#605C5C",]
# -----------------------------------
if PARAM_INDEX is None:
    DIR = "MultiTask"
else:
    DIR = f"Task{PARAM_INDEX}"

if LIMITED_MEMORY:
    MEM = "LimitedMemory"
    MEM_STR = "limited memory"
else:
    MEM = "UnlimitedMemory"
    MEM_STR = "unlimited memory"

if DWA_MODE == "Std":
    DWA_STR = "standard weighting"
elif DWA_MODE == "Norm1":
    DWA_STR = "sum-to-one weighting"
else:
    DWA_STR = "sum-to-K weighting"

COLUMNS_INTRA = [
    f"$\\mathcal{{L}}[u^{{\\theta^*}}, u]$",

    f"$\\mathcal{{L}}[\\nabla_x u^{{\\theta^*}}, \\nabla_x u]$",

    f"$\\mathcal{{L}}[\\nabla^2_x u^{{\\theta^*}}, \\nabla^2_x u]$",
    
    f"$\\mathcal{{L}}[\\mathcal{{G}}[u^{{\\theta^*}}], \\mathcal{{G}}[u]]$"
]

COLUMNS_INTER = [
    f"$\\mathcal{{L}}[u^{{\\theta^*}}, u]$",

    f"$\\mathcal{{L}}[\\nabla_x u^{{\\theta^*}}, \\nabla_x u]$",

    f"$\\mathcal{{L}}[\\nabla^2_x u^{{\\theta^*}}, \\nabla^2_x u]$",
    
    f"$\\mathcal{{L}}[\\mathcal{{G}}[u^{{\\theta^*}}], \\mathcal{{G}}[u]]$"
]

INDEXES = [
    f"Joint",
    f"Forget",
    f"EWC",
    f"Replay",
    f"Distil$^\\mathbf{{0}}$",
    f"Distil$^\\mathbf{{1}}$",
    f"Distil$^\\mathbf{{2}}$",
    f"Distil$^\\mathbf{{0, 1}}$",
    f"Distil$^\\mathbf{{0, 1, 2}}$",
]

In [2]:
def subsample(
    datasets: List[ConcatDataset],
    n_samples: int,
    seed: int = 42
    ) -> List[ConcatDataset]:
    
    reduced_datasets = []
    last_seed = seed
    for concat_ds in datasets:
        seeds = [last_seed+i for i in range(len(concat_ds))]
        last_seed = seeds[-1]
        reduced_concat_ds = []
        for ds, seed in zip(concat_ds.datasets, seeds):
            torch.manual_seed(seed)
            indices = torch.randperm(len(ds))
            indices = indices[:n_samples]
            reduced_cols = [col[indices] for col in ds.tensors]
            reduced_concat_ds.append(TensorDataset(*reduced_cols))
        reduced_datasets.append(ConcatDataset(reduced_concat_ds))
    return reduced_datasets

def merge_ds(datasets: List[TensorDataset]) -> TensorDataset:
    cols = None   
    for ds in datasets:
        new_cols = list(ds.tensors)
        if cols is None:
            cols = new_cols
        else:
            for i, col in enumerate(new_cols):
                cols[i] = torch.cat([cols[i], col])
    return TensorDataset(*cols)

def prepare_dataset(datasets: List[ConcatDataset]) -> ConcatDataset:
    n_snapshots = len(datasets[0].datasets)
    data = [None for _ in range(n_snapshots)]
    for i in range(n_snapshots):
        data[i] = merge_ds([concat_ds.datasets[i] for concat_ds in datasets])
    return ConcatDataset(data)

def make_bold_min(row):
    is_min = row == row.min()
    return["font-weight: bold !important; color: darkred !important;" if v else '' for v in is_min]

def to_scientific(val):
    if pd.isna(val) or isinstance(val, str):
        return val
    # Format to scientific notation with 2 decimal places
    return "{:.2e}".format(val)

def latex_sci(val):
    if pd.isna(val) or val == 0:
        return f"{val}"
    # Format to sci notation
    val = "{:.2e}".format(val)
    base, exponent = val.split("e")
    # Remove leading zeros and plus signs from exponent
    exponent = int(exponent) 
    return f"${base} \\times 10^{{{exponent}}}$"

def latex_bold_formatter(x, is_bold):
    if is_bold:
        return f"\\mathbf{{{latex_sci(x)}}}"
    return latex_sci(x)

def strip_math(s):
    return s[1:-1] if s.startswith("$") and s.endswith("$") else s

def get_hex_gradient(name: str, n_colors: int) -> List[str]:
    cmap = plt.get_cmap(name)
    
    # Generate n_colors evenly spaced values between 0 and 1
    # We use colors from 0.2 to 0.8 to avoid pure white/black extremes
    # Or use np.linspace(0, 1, n_colors) for the full range
    colors = [cmap(i) for i in np.linspace(0.2, 0.8, n_colors)]
    
    # Convert RGBA to Hex (dropping the alpha channel)
    return [mcolors.to_hex(c).replace('#', '').upper() for c in colors]

In [3]:
full_datasets = torch.load(f"{PDE}/data/full_datasets.pth", weights_only=False).datasets
unlabeled_datasets = torch.load(f"{PDE}/data/unlabeled_datasets.pth", weights_only=False).datasets
dev_datasets = torch.load(f"{PDE}/data/dev_datasets.pth", weights_only=False).datasets
train_datasets = torch.load(f"{PDE}/data/train_datasets.pth", weights_only=False).datasets
val_datasets = torch.load(f"{PDE}/data/val_datasets.pth", weights_only=False).datasets
inter_test_datasets = torch.load(f"{PDE}/data/inter_test_datasets.pth", weights_only=False).datasets
intra_test_datasets = torch.load(f"{PDE}/data/intra_test_datasets.pth", weights_only=False).datasets

n_params = 2 # number of xi_j

if PARAM_INDEX != None:
    train_dataset = [dev_datasets[PARAM_INDEX]]
    train_boundary_dataset = [ConcatDataset([extract_boundary(dev_datasets[PARAM_INDEX])])]
    unlabeled_dataset = [unlabeled_datasets[PARAM_INDEX]]
    intra_test_dataset = [intra_test_datasets[PARAM_INDEX]]
else:
    train_dataset = dev_datasets
    train_boundary_dataset = [ConcatDataset([extract_boundary(ds)]) for ds in dev_datasets]
    unlabeled_dataset = unlabeled_datasets[:len(dev_datasets)]
    intra_test_dataset = intra_test_datasets

train_data = prepare_dataset(train_dataset)
intra_test_data = prepare_dataset(intra_test_dataset).datasets[0]
inter_test_data = prepare_dataset(inter_test_datasets).datasets[0]

# Global performances

In [4]:
colors = [{
    "index": ORANGE[3][1:], 
    "columns": ORANGE[4][1:],
    "highlight": ORANGE[0][1:]
},
{
    "index": BLUE[2][1:], 
    "columns": BLUE[3][1:],
    "highlight": BLUE[0][1:]
}]

#oranges = get_hex_gradient("YlOrRd", 10)
#blues   = get_hex_gradient("Blues", 10)
#purples = get_hex_gradient("autumn", 10)

oranges = ["FFB780", "FFBB56", "FFD981", "FFF2BE"]
blues   = ["99B1FF", "B2C3FD", "D4DEFF", "ECF5FF"]
purples = ["9932CC", "BA55D3", "D8BFD8", "E6E6FA", "F8F8FF"]

#oranges.reverse()
#blues.reverse()
#purples.reverse()

color_scales = {
    "orange": oranges,
    "blue": blues,
    "purple": purples
}

# Choose which one you want to use for this specific table
gradients = [color_scales["orange"], color_scales["blue"]]

from_scratch = False
fine_tuning = True

full_model = f"{PDE}/FullDomainLearning/{DIR}/{ACTUAL_MODE}_{DWA_MODE}_{CLIP}_{N_EPOCHS}/models2/trial0/model.pth"
forget_models = {
    "FromScratch": f"{PDE}/DomainIncrementalLearning/{DIR}/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}{BD}/Forget/FromScratch{N_EPOCHS}/Dom3/models/trial0/model.pth",
    "FineTune": f"{PDE}/DomainIncrementalLearning/{DIR}/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}{BD}/Forget/FineTune{N_EPOCHS}/Dom3/models/trial0/model.pth"
}
ewc_models = {
    #"FromScratch": f"{PDE}/DomainIncrementalLearning/{DIR}/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}{BD}/{MEM}/EWC/alpha_0.5/weight_auto/warmup1_decay1.0/FromScratchLongMemory{N_EPOCHS}/Dom3/models/trial0/model.pth",
    #"FineTune": f"{PDE}/DomainIncrementalLearning/{DIR}/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}{BD}/{MEM}/EWC/alpha_0.5/weight_auto/warmup1_decay1.0/FineTuneLongMemory{N_EPOCHS}/Dom3/models/trial0/model.pth",
    "FineTune": f"{PDE}/DomainIncrementalLearning/{DIR}/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}{BD}/{MEM}/EWC/alpha_1.0/weight_auto/warmup1_decay1.0/FineTuneLongMemory{N_EPOCHS}/Dom3/models/trial0/model.pth"
}
replay_models = {
    "FromScratch": f"{PDE}/DomainIncrementalLearning/{DIR}/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}{BD}/{MEM}/Replay/Residual+Boundary/FromScratch{MEMORY}{N_EPOCHS}/Dom3/models/trial0/model.pth",
    "FineTune": f"{PDE}/DomainIncrementalLearning/{DIR}/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}{BD}/{MEM}/Replay/Residual+Boundary/FineTune{MEMORY}{N_EPOCHS}/Dom3/models/trial0/model.pth"
}
distill_modes = ["Output", "Derivative", "Hessian", "Output+Derivative", "Output+Derivative+Hessian"]
distill_models = {}
for mode in distill_modes:
    distill_models[mode] = {
        "FromScratch": f"{PDE}/DomainIncrementalLearning/{DIR}/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}{BD}/{MEM}/Distill/{mode}/FromScratch{MEMORY}{N_EPOCHS}/Dom3/models/trial0/model.pth",
        "FineTune": f"{PDE}/DomainIncrementalLearning/{DIR}/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}{BD}/{MEM}/Distill/{mode}/FineTune{MEMORY}{N_EPOCHS}/Dom3/models/trial0/model.pth"
    }

#distill_modes2 = ["Out", "Der1", "Der2", "Out+Der1", "Out+Der1+Der2"]
#distill_ids = []
#distill_models2 = []
#for mode, mode2 in zip(distill_modes, distill_modes2):
#    if from_scratch:
#        distill_ids.append(f"Distill {mode2} FS")
#        distill_models2.append(distill_models[mode]["FromScratch"])
#    if fine_tuning:
#        distill_ids.append(f"Distill {mode2} FT")
#        distill_models2.append(distill_models[mode]["FineTune"])
distill_models2 = []
for mode in distill_modes:
    if from_scratch:
        distill_models2.append(distill_models[mode]["FromScratch"])
    if fine_tuning:
        distill_models2.append(distill_models[mode]["FineTune"])
modes = []
if from_scratch:
    modes.append("FS")
if fine_tuning:
    modes.append("FT")

if len(modes) == 1:
    row_ids = []
    for r in INDEXES:
        row_ids.append(f"\\textbf{{{r}}}")
        #\
        #["Full Learning"] + \
        #[f"Forget"] + \
        #[f"EWC"] + \
        #[f"Replay Res+BC"] + \
        #distill_ids
else:
    row_ids = []
    for i, r in enumerate(INDEXES):
        for s in modes:
            if i > 0:
                row_ids.append(f"\\textbf{{{r}}} {s}")
            else:
                row_ids.append(f"\\textbf{{{r}}}")
modes = []
if from_scratch:
    modes.append("FromScratch")
if fine_tuning:
    modes.append("FineTune")

models = \
    [full_model] + \
    [forget_models[s] for s in modes] + \
    [ewc_models["FineTune"]] + \
    [replay_models[s] for s in modes] + \
    distill_models2

datasets = [(colors[0], gradients[0], COLUMNS_INTRA, intra_test_data, "Intra Test"), (colors[1], gradients[1], COLUMNS_INTER, inter_test_data, "Inter Test")]

for color, gradient, columns, ds, ds_name in datasets:
    rows = []
    for model_file in models:
        model = resume_model(model_path=model_file)
        rows.append(model.evaluate(ds))
    
    df = pd.DataFrame(
        data=rows,
        columns=columns,
        index=row_ids
    )

    df.index = [f"\\color[HTML]{{{color['index']}}}{{{x}}}" for x in df.index]
    df.columns = [f"\\color[HTML]{{{color['columns']}}}\\textbf{{{x}}}" for x in df.columns]
    
    # Calculate rank per column (excluding the first row)
    ranks = df.iloc[1:].rank(axis=0, method='min').astype(int)
    #ranks = df.rank(axis=0, method='min').astype(int)
    
    df_latex = df.copy()
    for i in df.index:
        for j in df.columns:
            val = df.loc[i, j]
            if i in ranks.index:
                r = ranks.loc[i, j]
            else:
                r = len(gradient)+1

            if r <= len(gradient):
                hex_color = gradient[r - 1]

                formatted_val = strip_math(latex_sci(val))
                if r == 1:
                    formatted_val = f"\\mathbf{{{formatted_val}}}"
                df_latex.loc[i, j] = f"\\cellcolor[HTML]{{{hex_color}}}${formatted_val}$"
            else:
                df_latex.loc[i, j] = latex_sci(val)
    latex_tabular = df_latex.to_latex(
        column_format="l" + "c" * len(df.columns),
        escape=False
    )
    mode = ""
    if not(from_scratch and fine_tuning):
        if from_scratch:
            mode = " (from scratch)"
        if fine_tuning:
            mode = " (fine tuning)"
    caption = f"{ds_name.lower()} global performances{mode}."
    mode = ""
    if from_scratch:
        mode += "FS"
    if fine_tuning:
        mode += "FT"
    label = f"{ds_name.replace(' ', '')}{mode}GlobalPerformances"
    latex_code = f"""
\\begin{{table}}[H]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
{latex_tabular}\\end{{table}}
        """
    with open(
        f"{PDE}/{ds_name.replace(' ', '')}DISummaryTable{N_EPOCHS}.tex",
        "w"
    ) as f:
        f.write(latex_code)

In [5]:
colors = [{
    "index": ORANGE[3][1:], 
    "columns": ORANGE[4][1:],
    "highlight": ORANGE[0][1:]
},
{
    "index": BLUE[2][1:], 
    "columns": BLUE[3][1:],
    "highlight": BLUE[0][1:]
}]
color_scales = {
    "orange": ["FFB780", "FFBB56", "FFD981", "FFF2BE"],
    "blue":   ["99B1FF", "B2C3FD", "D4DEFF", "ECF5FF"],
    "purple": ["9932CC", "BA55D3", "D8BFD8", "E6E6FA", "F8F8FF"]
}

# Choose which one you want to use for this specific table
gradients = [color_scales["orange"], color_scales["blue"]]

from_scratch = False
fine_tuning = True

full_model = f"{PDE}/FullDomainLearning/{DIR}/{ACTUAL_MODE}_{DWA_MODE}_{CLIP}_{N_EPOCHS}/models2/trial0/model.pth"
forget_models = {
    "FromScratch": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/Forget/FromScratch{N_EPOCHS}/Task3/models/trial0/model.pth",
    "FineTune": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/Forget/FineTune{N_EPOCHS}/Task3/models/trial0/model.pth"
}
ewc_models = {
    #"FromScratch": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/EWC/alpha_0.5/weight_auto/warmup1_decay1.0/FromScratchLongMemory{N_EPOCHS}/Task3/models/trial0/model.pth",
    "FineTune": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/EWC/alpha_1.0/weight_auto/warmup1_decay1.0/FineTuneLongMemory{N_EPOCHS}/Task3/models/trial0/model.pth"
}
replay_models = {
    "FromScratch": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/Replay/Residual+Boundary/FromScratch{MEMORY}{N_EPOCHS}/Task3/models/trial0/model.pth",
    "FineTune": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/Replay/Residual+Boundary/FineTune{MEMORY}{N_EPOCHS}/Task3/models/trial0/model.pth"
}
distill_modes = ["Output", "Derivative", "Hessian", "Output+Derivative", "Output+Derivative+Hessian"]
distill_models = {}
for mode in distill_modes:
    distill_models[mode] = {
        "FromScratch": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/Distill/{mode}/FromScratch{MEMORY}{N_EPOCHS}/Task3/models/trial0/model.pth",
        "FineTune": f"{PDE}/TaskIncrementalLearning/{ACTUAL_MODE}{LR}_{DWA_MODE}_{CLIP}/{MEM}/Distill/{mode}/FineTune{MEMORY}{N_EPOCHS}/Task3/models/trial0/model.pth"
    }

distill_models2 = []
for mode in distill_modes:
    if from_scratch:
        distill_models2.append(distill_models[mode]["FromScratch"])
    if fine_tuning:
        distill_models2.append(distill_models[mode]["FineTune"])
modes = []
if from_scratch:
    modes.append("FS")
if fine_tuning:
    modes.append("FT")

modes = []
if from_scratch:
    modes.append("FS")
if fine_tuning:
    modes.append("FT")
if len(modes) == 1:
    row_ids = []
    for r in INDEXES:
        row_ids.append(f"\\textbf{{{r}}}")
else:
    row_ids = []
    for i, r in enumerate(INDEXES):
        for s in modes:
            if i > 0:
                row_ids.append(f"\\textbf{{{r}}} {s}")
            else:
                row_ids.append(f"\\textbf{{{r}}}")
modes = []
if from_scratch:
    modes.append("FromScratch")
if fine_tuning:
    modes.append("FineTune")
#[ewc_models[s] for s in modes] + \
models = \
    [full_model] + \
    [forget_models[s] for s in modes] + \
    [ewc_models["FineTune"]] + \
    [replay_models[s] for s in modes] + \
    distill_models2

datasets = [(colors[0], gradients[0], COLUMNS_INTRA, intra_test_data, "Intra Test"), (colors[1], gradients[1], COLUMNS_INTER, inter_test_data, "Inter Test")]

for color, gradient, columns, ds, ds_name in datasets:
    rows = []
    for model_file in models:
        model = resume_model(model_path=model_file)
        rows.append(model.evaluate(ds))
    
    df = pd.DataFrame(
        data=rows,
        columns=columns,
        index=row_ids
    )

    df.index = [f"\\color[HTML]{{{color['index']}}}{{{x}}}" for x in df.index]
    df.columns = [f"\\color[HTML]{{{color['columns']}}}\\textbf{{{x}}}" for x in df.columns]
    
    ranks = df.iloc[1:].rank(axis=0, method='min').astype(int)

    df_latex = df.copy()
    for i in df.index:
        for j in df.columns:
            val = df.loc[i, j]
            if i in ranks.index:
                r = ranks.loc[i, j]
            else:
                r = len(gradient)+1

            if r <= len(gradient):
                hex_color = gradient[r - 1]
                formatted_val = strip_math(latex_sci(val))
                if r == 1:
                    formatted_val = f"\\mathbf{{{formatted_val}}}"
                df_latex.loc[i, j] = f"\\cellcolor[HTML]{{{hex_color}}}${formatted_val}$"
            else:
                df_latex.loc[i, j] = latex_sci(val)

    latex_tabular = df_latex.to_latex(
        column_format="l" + "c" * len(df.columns),
        escape=False
    )
    mode = ""
    if not(from_scratch and fine_tuning):
        if from_scratch:
            mode = " (from scratch)"
        if fine_tuning:
            mode = " (fine tuning)"
    caption = f"{ds_name.lower()} global performances{mode}."
    mode = ""
    if from_scratch:
        mode += "FS"
    if fine_tuning:
        mode += "FT"
    label = f"{ds_name.replace(' ', '')}{mode}GlobalPerformances"
    latex_code = f"""
\\begin{{table}}[H]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
{latex_tabular}\\end{{table}}
        """
    with open(
        f"{PDE}/{ds_name.replace(' ', '')}TISummaryTable{N_EPOCHS}.tex",
        "w"
    ) as f:
        f.write(latex_code)